In [ ]:
# Packages
import scanpy as sc
import squidpy as sq
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib
import os
import scipy
import scipy.sparse
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings

# Suppress Pandas and Squidpy warnings
pd.options.mode.chained_assignment = None
warnings.filterwarnings("ignore", category=UserWarning, module='squidpy')

In [ ]:
# Set paths
out_dir = ''
obj_dir = ''

sc.settings.figdir = out_dir
sc.set_figure_params(dpi=100, dpi_save=400, figsize=(6,6))

### Transferring c2l labels to full object

In [ ]:
# Read in c2l annotated object and full object
adata_c2l = sc.read_h5ad(f'{obj_dir}orv01_spatial_c2l_annotated.h5ad')
adata_c2l

In [ ]:
adata = sc.read_h5ad(f'{obj_dir}orv01_spatial_clustered.h5ad')
adata

In [ ]:
# Transfer labels
cols_to_transfer = ['+Prolif B cells', '+Ribo/-Prolif B cells', '-Ribo B cells', 'CD4+ T cells', 'CD8+ T cells',
    'IL23R+ T cells', 'Myeloid cells', 'NK cells', 'Naive B cells', 'Naive T cells', 'Proliferating T cells', 'c2l_predicted',
    'grouped_annotations']

adata.obs[cols_to_transfer] = adata_c2l.obs.loc[adata.obs_names, cols_to_transfer]

In [ ]:
adata

In [ ]:
# Ensure you can plot properly
sq.pl.spatial_scatter(adata, color='-Ribo B cells', size=1, figsize=(5,5), cmap='viridis')

In [ ]:
sq.pl.spatial_scatter(adata, color='grouped_annotations')

In [ ]:
# Save
adata.write(f'{obj_dir}orv01_spatial_clustered.h5ad')

In [ ]:
# Read back in
adata = sc.read_h5ad(f'{obj_dir}orv01_spatial_clustered.h5ad')
adata

### Moran's I scoring

Calculate with all genes

In [ ]:
# Confirming log-normalized counts
adata.X.min(), adata.X.max()

In [ ]:
# Check if hvgs are calculated
hvg_genes = adata.var_names[adata.var["highly_variable"]]

In [ ]:
# Calculate SVGs on HVGs
sq.gr.spatial_autocorr(adata, mode="moran", genes=hvg_genes)

In [ ]:
# Grab top 50 genes
df = adata.uns['moranI']
df.head()

In [ ]:
# Make gene names a column
df.index.name = 'gene'
df.reset_index(inplace=True)
df

In [ ]:
# Save
df.to_csv(f'{out_dir}moran_scores_svg.csv', index=False)

Calculate with ribosomal genes removed from HVGs

In [ ]:
# Remove ribosomal genes and recalculate
hvg_no_ribo = hvg_genes[~hvg_genes.str.startswith(("RPS", "RPL"))]

In [ ]:
len(hvg_no_ribo)

In [ ]:
# Recompute
moran_df_no_ribo = sq.gr.spatial_autocorr(adata, mode="moran", genes=hvg_no_ribo, copy=True)

In [ ]:
moran_df_no_ribo.head(n=20)

Calculate with ribosomal genes removed BEFORE computing HVGs

In [ ]:
# Remove ribo genes
non_ribo_mask = ~adata.var_names.str.startswith(("RPS", "RPL"))
adata_no_ribo = adata[:, non_ribo_mask].copy()

In [ ]:
sc.pp.highly_variable_genes(
    adata_no_ribo,
    flavor="seurat",
    n_top_genes=3000)

In [ ]:
hvg_genes_no_ribo_2 = adata_no_ribo.var_names[
    adata_no_ribo.var["highly_variable"]]

In [ ]:
moran_df_no_ribo_2 = sq.gr.spatial_autocorr(
    adata,
    genes=hvg_genes_no_ribo_2,
    mode="moran", copy=True)

In [ ]:
moran_df_no_ribo_2.head(n=15)

Results appear robust to these variations

In [ ]:
top_genes = list(df['gene'].head(100))
top_genes

In [ ]:
sq.pl.spatial_scatter(adata, color='CD79B', size=1, figsize=(5,5), cmap='plasma')


In [ ]:
# Plotting the top SVGs
sq.pl.spatial_scatter(adata, color=['RPL37A', 'RPS20', 'COL3A1', 'RPS27', 'COL1A1', 'RPS12', 'SPARC'], size=1, figsize=(5,5), cmap='plasma')

In [ ]:
# Plotting some additional SVGs of interest
sq.pl.spatial_scatter(adata, color=['C3', 'APOE', 'CD248', 'CCN2', 'FN1'], size=1, figsize=(5,5), cmap='plasma')

In [ ]:
# Plotting genes relevant to the -Ribo B population
sq.pl.spatial_scatter(adata, color=['DOCK8', 'TNFAIP8', 'KMT2C',
                                   'SLC41A2', 'FBXW7', 'PCCA', 'CAMKMT'], size=1, figsize=(5,5), cmap='plasma')

In [ ]:
sq.pl.spatial_scatter(adata, color='IGFBP7', size=1, figsize=(5,5), cmap='plasma')

In [ ]:
sq.pl.spatial_scatter(adata, color='S100A6', size=1, figsize=(5,5), cmap='plasma')

In [ ]:
sc.set_figure_params(dpi=100, dpi_save=400, figsize=(7,7))

# Plot top 10 genes
top_genes = top_genes[0:21]

for svg in top_genes:
    fig, ax = plt.subplots(figsize=(7,7))
    sq.pl.spatial_scatter(adata, color=svg, size=1, figsize=(5,5), cmap='plasma', ax=ax)
    plt.savefig(f'{out_dir}{svg}_spatial_plot.png')

In [ ]:
# Other genes of interest
svgs = ['CD79B', 'CD68', 'CD163', 'CD4', 'CD8A', 'MKI67']

for svg in svgs:
    fig, ax = plt.subplots(figsize=(7,7))
    sq.pl.spatial_scatter(adata, color=svg, size=1, figsize=(5,5), cmap='plasma', ax=ax)
    plt.savefig(f'{out_dir}{svg}_spatial_plot.png')